In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program for a multi-agent flocking simulation (boids). The input consists of:
</p>
  <li>An array <code>agents</code> containing <code>N</code> agents, where <code>N</code> is the total number of agents</li>
  <li>Each agent occupies 4 consecutive 32-bit floating point numbers in the array: $[x, y, v_x, v_y]$, where:
      <ul>
          <li>$(x, y)$ represents the agent's position in 2D space</li>
          <li>$(v_x, v_y)$ represents the agent's velocity vector</li>
      </ul>
  </li>
  <li>The total array size is <code>4 * N</code> floats, with agent $i$'s data stored at indices <code>[4i, 4i+1, 4i+2, 4i+3]</code></li>
</ul>

<h2>Simulation Rules</h2>
<ol>
  <li>For each agent $i$, identify all neighbors $j$ (where $i \neq j$) within radius $r = 5.0$ using:
      $$
      \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} < r
      $$
  </li>
  <li>Compute average velocity of neighboring agents:
      $$
      \vec{v}_{avg} = \begin{cases}
      \frac{1}{|N_i|} \sum_{j \in N_i} \vec{v}_j & \text{if } |N_i| > 0 \\
      \vec{v}_i & \text{if } |N_i| = 0
      \end{cases}
      $$
      where $N_i$ is the set of neighbors for agent $i$
  </li>
  <li>Update velocity:
      $$
      \vec{v}_{new} = \vec{v} + \alpha(\vec{v}_{avg} - \vec{v}), \text{ where } \alpha = 0.05
      $$
  </li>
  <li>Update position:
      $$
      \vec{p}_{new} = \vec{p} + \vec{v}_{new}
      $$
  </li>
</ol>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>agents_next</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: N = 2
agents = [
  0.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 4.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]

Output:
agents_next = [
  1.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 3.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]
</pre>

<h2>Constraints</h2>
<ul>
<li>1 &le; <code>N</code> &le; 100,000</li>
<li>Each agent's position and velocity components are 32-bit floats</li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// agents, agents_next are device pointers
extern "C" void solve(const float* agents, float* agents_next, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# agents, agents_next are tensors on the GPU
@cute.jit
def solve(agents: cute.Tensor, agents_next: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# agents is a tensor on the GPU
@jax.jit
def solve(agents: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    agents: UnsafePointer[Float32, MutExternalOrigin],
    agents_next: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# agents, agents_next are tensors on the GPU
def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# agents, agents_next are tensors on the GPU
def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/hard/14_multi_agent_sim/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
